In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import MDAnalysis as mda
import numpy as np
from MDAnalysisTests.datafiles import DCD, PSF

from mdadash.backend.analyses.com_distance import COMDistance
from mdadash.backend.kernel.core import BufferedTrajectory

u = mda.Universe(PSF, DCD)

buffer_size = 50
u.trajectory = BufferedTrajectory(u, buffer_size, use_shm=False)

c = COMDistance()
c.u = u
# vectorization is only supported in batch mode if trajectory is already
# unwrapped. Disable unwrap to ensure we comparing correctly
c.unwrap = False
c.selection1 = "resid 1"
c.selection2 = "protein"
c.on_post_connect()
for i in range(buffer_size - 1):
    u.trajectory._prepare_for_next()
    u.trajectory.next()

box = "triclinic"

if box == "ortho":
    unitcells = np.array([60.0, 60.0, 60.0, 90.0, 90.0, 90.0])
elif box == "triclinic":
    unitcells = np.array([60.0, 60.0, 60.0, 60.0, 60.0, 60.0])
else:
    unitcells = np.array([0, 0, 0, 0, 0, 0])
u.trajectory._unitcells = np.repeat(
    unitcells[np.newaxis, :], len(u.trajectory._unitcells), axis=0
)

In [ ]:
# without vectorization
c.batch_vectorization = False
non_vec_values = c._compute_batch()
%timeit c._compute_batch()

14.2 ms ± 234 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# with vectorization
c.batch_vectorization = True
vec_values = c._compute_batch()
%timeit c._compute_batch()

3.62 ms ± 153 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# ensure computed values are the same
np.allclose(non_vec_values, vec_values, rtol=1e-05, atol=1e-05)

True

In [ ]:
c._reset_plot_values()
c._update_plot(non_vec_values)

In [ ]:
c._reset_plot_values()
c._update_plot(vec_values)